# Introdução: 
O preço das passagens aéreas muda o tempo todo e pode variar bastante, o faz esse tema ser interessante para tentar predizer o preço através de outros fatores. No nosso Dataset temos informações sobre vôos entre diversas cidades da Índia, sendo os preditores: a companhia aérea, o trajeto entre as cidades de origem e destino, a classe escolhida, o número de escalas, a duração do voo e a antecedência com que a passagem é comprada. O desafio é entender como cada um desses aspectos pesa na hora de definir quanto o passageiro vai pagar no final.



# Análise Exploratória: 


## Setup

In [ ]:
# Importando bibliotecas

import numpy as np
import pandas as pd
import altair as alt
import vegafusion
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import geodatasets
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, get_cmap


### Importando base de dados (Flight)

In [ ]:
df_flight = pd.read_csv("flight.csv")

df_flight.head()

## Limpeza e Qualidade dos dados

In [ ]:
# Removendo a primeira coluna, pois não acrescenta na análise

df_flight = df_flight.drop('Unnamed: 0',axis=1)

A primeira coluna, Unnamed: 0, é um índice redundante que não adiciona valor preditivo ou contextual à nossa análise. Para garantir a limpeza e foco do dataset, optamos por remover essa coluna antes de prosseguir com a análise descritiva e visual.

In [ ]:
# Informações
df_flight.info()

# Verificar valores nulos
df_flight.isnull().sum()

Aqui podemos observar que não temos valores nulos a serem tratados.

## Colunas

As colunas existentes no dataset escolhido são:  

1. **airline** – Companhia aérea responsável pelo voo.  

2. **flight** – Código ou número identificador do voo.  

3. **source_city** – Cidade de origem de onde o voo parte.  

4. **departure_time** – Faixa de horário da decolagem (por exemplo: manhã, tarde, noite).  

5. **stops** – Número de paradas intermediárias (ex: zero, one, two_or_more).  

6. **arrival_time** – Faixa de horário de chegada ao destino.  

7. **destination_city** – Cidade de destino para onde o voo está indo.  

8. **class** – Classe da passagem (por exemplo: Economy, Business).  

9. **duration** – Duração total do voo em horas.  

10. **days_left** – Quantidade de dias restantes até a data do voo no momento da coleta dos dados.  

11. **price** – Preço da passagem aérea.


Com a estrutura e qualidade dos dados verificadas, o foco se volta para as variáveis que servirão como preditores do preço da passagem (price). A compreensão de cada uma é crucial para guiar as visualizações e o futuro modelo de predição.

## Análise unidimensional

Para ter uma visão inicial da distribuição e das tendências centrais das variáveis numéricas, geramos o resumo estatístico que detalha a contagem, média, desvio padrão, valores mínimos/máximos e os quartis.

In [ ]:
df_flight.describe()

A análise do resumo estatístico mostra:

* duration (Duração): Os voos têm uma duração média de 11,8 horas, com um desvio padrão alto, indicando uma grande variabilidade no tempo de viagem, provavelmente devido à inclusão de paradas.

* days_left (Antecedência): As passagens são compradas, em média, com 26 dias de antecedência. A distribuição parece ser relativamente uniforme, conforme sugerido pelo desvio padrão em relação ao intervalo.

* price (Preço): O preço médio é de aproximadamente 19.865 (R$), com um desvio padrão que sugere uma dispersão considerável e a possível presença de outliers, o que será melhor visualizado nos boxplots.

#### Histogramas

In [ ]:
# Pela quantidade de observações, é necessário desabilitar o máximo de colunas padrão do Altair (5000)

alt.data_transformers.disable_max_rows()


##### Histogramas das variáveis numéricas

A seguir, visualizamos a distribuição de frequência das variáveis numéricas através de histogramas no Altair, uma técnica que permite identificar a forma da distribuição, o centro e a dispersão dos dados. É particularmente útil para verificar se as variáveis se aproximam de uma distribuição normal ou se há assimetrias.

In [ ]:
num_cols = ["duration", "days_left", "price"]

charts = [
    alt.Chart(df_flight).mark_bar().encode(
        alt.X(col, type="quantitative", bin=alt.BinParams(maxbins=15), title=col),
        alt.Y("count()", type="quantitative", title="Contagem de ocorrências")
    ).properties(width=150, height=150)
    for col in num_cols
]

histogramas_num = alt.hconcat(*charts).properties(
    title="Distribuição de variáveis numéricas"
)

histogramas_num

Os histogramas confirmam a dispersão observada:

* duration: Mostra uma distribuição aparentemente normal.

* days_left: Apresenta uma distribuição relativamente plana (uniforme), o que corrobora a análise descritiva sobre a homogeneidade na antecedência de compra.

* price: Exibe uma distribuição positivamente assimétrica (à direita), com a maioria dos preços concentrada em valores mais baixos e uma cauda longa para valores mais altos, indicando a presença de voos premium ou de longa duração.

##### Histogramas das variáveis categóricas

A análise de variáveis categóricas continua com histogramas de frequência, que nos permitem avaliar a distribuição das observações entre as diferentes categorias de cada preditor, como airline, class e source_city.

In [ ]:
cat_cols = [
    "airline", "source_city", "departure_time", "stops",
    "arrival_time", "destination_city", "class"
]
charts_cat = [
    alt.Chart(df_flight).mark_bar().encode(
        x=alt.X(col, type="nominal", title=col, sort='-y'),
        y=alt.Y("count()", type="quantitative", title="Contagem de ocorrências"),
        tooltip=[col, "count()"]
    ).properties(width=150, height=150)
    for col in cat_cols
]

# organiza os gráficos em uma grade
histogramas_cat = alt.hconcat(*charts_cat[:4]).resolve_scale(y='independent')
if len(cat_cols) > 4:
    row2 = alt.hconcat(*charts_cat[4:]).resolve_scale(y='independent')
    histogramas_cat = alt.vconcat(histogramas_cat, row2)

histogramas_cat = histogramas_cat.properties(
    title="Distribuição de variáveis categóricas"
)

histogramas_cat


Os histogramas categóricos revelam a proporção de dados por categoria, sendo útil para identificar desequilíbrios que podem influenciar modelos preditivos:

* airline: Vistara é a companhia aérea com o maior número de voos no dataset.

* class: Há um claro desequilíbrio entre as classes, com Economy dominando a maioria das observações.

* source_city e destination_city: Delhi e Mumbai são as cidades mais representadas, tanto como origem quanto como destino.

#### Boxplots para as variáveis numéricas

Para complementar a análise univariada das variáveis numéricas, utilizamos boxplots. Estes gráficos são ideais para identificar a dispersão, a simetria dos dados e, principalmente, a presença e extensão de outliers, que podem impactar a robustez do modelo.

In [ ]:
col_titles = {
    "duration": "Duração do voo (horas)",
    "days_left": "Dias restantes até o voo",
    "price": "Preço da passagem (R$)"
}


plt.figure(figsize=(15, 4))

# cria um boxplot horizontal para cada variável numérica
for i, col in enumerate(num_cols):
    plt.subplot(1, 3, i+1) 
    sns.boxplot(x=df_flight[col])  
    plt.title(f"Boxplot de {col_titles[col]}")
    plt.grid(axis="x", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

Os boxplots fornecem informações visuais sobre a dispersão:

* duration: A mediana está em torno de 10 horas, mas há uma grande quantidade de outliers na cauda superior, confirmando a alta variabilidade e a concentração de voos de longa duração.

* days_left: A distribuição é bem centralizada e simétrica, com poucos outliers extremos, reforçando a uniformidade já percebida.

* price: A distribuição está concentrada na parte inferior, com uma extensa cauda de outliers (preços muito altos), o que é esperado para passagens de classe Business ou compras de última hora. Este é um indicativo de que o preço é a variável mais sensível a valores extremos no dataset.

## Análise Bidimensional


A análise bidimensional é crucial para entender as relações e a correlação entre os preditores e a variável alvo, o price. Começamos examinando a relação entre as variáveis numéricas.

Vamos começar por um mapa de calor entre as variáveis numéricas:

In [ ]:
num_cols = ["duration", "days_left", "price"]

plt.figure(figsize=(10, 6))
sns.heatmap(df_flight[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Mapa de Calor - Correlação entre variáveis numéricas')
plt.show()

O mapa de calor de correlação revela a seguinte relação:
* duration vs. price: Apresentam uma correlação moderada-positiva ($0.20$). Embora a duração do voo (incluindo escalas) não seja o principal fator, voos mais longos tendem a ser um pouco mais caros.


Agora prosseguimos para tentar ver as relações visualmente através de scaterplots:

In [ ]:
sns.pairplot(
    df_flight[num_cols],
    diag_kind="kde",          
    corner=True,              
    plot_kws={'alpha': 0.6, 's': 40, 'edgecolor': 'none'},
    diag_kws={'fill': True, 'color': '#4C72B0'}
)

plt.suptitle("Scatter Plot das Variáveis Numéricas", y=1.02, fontsize=14)
plt.show()


O pairplot anterior com todos os dados não permitiu identificar padrões claros. Para melhorar a visualização das interações, geramos um gráfico de dispersão (scatter plot) com uma amostra de 1.000 observações, colorido pela airline (companhia aérea). Embora ainda haja dispersão, este plot com amostra nos dá um vislumbre das faixas de preço e duração por companhia.

In [ ]:
df_sample = df_flight.sample(n=1000, random_state=42)

# --- Gerar o pairplot com estilo limpo ---
sns.pairplot(
    df_sample,
    vars=num_cols,
    hue="airline",
    diag_kind="kde",
    corner=True,
    plot_kws={'alpha': 0.6, 's': 35, 'edgecolor': 'none'}
)

plt.suptitle("Scatter Plot das Variáveis Numéricas (Amostra de 1.000 voos)", y=1.02, fontsize=14)
plt.show()

A partir desse gráfico, é interessante buscar o padrão de preços por companhia aérea. O boxplot a seguir visualiza a distribuição de preços para cada companhia, permitindo comparar não apenas as medianas, mas também a dispersão e a presença de outliers.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_flight, x='airline', y='price',color= 'red')
plt.title('Distribuição do Preço por Companhia Aérea')
plt.xticks(rotation=45)
plt.show()


O boxplot de preço por Companhia Aérearevela uma diferença acentuada nas políticas de preço:

* Vistara é a companhia com a maior mediana de preços e também com a maior dispersão (amplitude interquartil).

* AirAsia e SpiceJet apresentam as menores medianas, confirmando a percepção de que são companhias de custo mais baixo no dataset.

* A presença de outliers na cauda superior para quase todas as companhias é evidente, indicando a venda de passagens Business ou rotas de alta demanda.

A hora do dia em que um voo parte ou chega pode ter um impacto direto no preço, devido à demanda e conveniência. Analisamos o preço médio por horário de partida (departure_time) e horário de chegada (arrival_time).

In [ ]:
departure_price = df_flight.groupby('departure_time')['price'].mean().reset_index()
departure_price

In [ ]:
arrival_price = df_flight.groupby('arrival_time')['price'].mean().reset_index()
arrival_price

In [ ]:

sns.set(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Partida ---
sns.boxplot(
    data=df_flight,
    x='departure_time',
    y='price',
    color=sns.color_palette("viridis", n_colors=1)[0],  # única cor
    ax=axes[0]
)
axes[0].set_title('Preço por Horário de Partida')
axes[0].set_xlabel('Horário de Partida')
axes[0].set_ylabel('Preço')
axes[0].grid(True, linestyle='--', alpha=0.3)

# --- Chegada ---
sns.boxplot(
    data=df_flight,
    x='arrival_time',
    y='price',
    color=sns.color_palette("magma", n_colors=1)[0],  # única cor
    ax=axes[1]
)
axes[1].set_title('Preço por Horário de Chegada')
axes[1].set_xlabel('Horário de Chegada')
axes[1].set_ylabel('')
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()


Agora vamos tentar juntar essas informações em uma única visualização. O preço é influenciado pela combinação dos horários de partida e chegada. O gráfico de linha ilustra essa interação, mostrando como o preço médio de chegada varia, segmentado pelo horário de partida. Isso ajuda a entender o valor cobrado pela conveniência de rotas específicas

In [ ]:
sns.set_theme(style="whitegrid")

g = sns.relplot(
    data=df_flight,
    x="arrival_time",
    y="price",
    col="departure_time",
    kind="line",
    marker="o",
    linewidth=2,
    height=4,
    aspect=1,
    palette="viridis"
)

# Ajustes estéticos
g.set_titles(col_template="Partida: {col_name}")
g.set_axis_labels("Horário de Chegada", "Preço Médio")
g.fig.suptitle("💸 Relação entre Preço e Horário de Chegada por Horário de Partida", 
               fontsize=14, fontweight="bold", y=1.05)

# Rotação dos rótulos do eixo X
for ax in g.axes.flatten():
    ax.tick_params(axis='x', rotation=45)

plt.show()

O gráfico de linha confirma a complexidade da relação Preço x Horário de Voo:

* Partidas 'Night' (Noite) e 'Morning' (Manhã): Estas rotas tendem a ter preços médios mais altos em quase todos os horários de chegada, sugerindo que o horário de partida é um forte preditor de preço.

* Voos 'madrugadores' ('Early_Morning'): As linhas para voos de manhã cedo (seja na partida ou na chegada) tendem a mostrar picos de preço em horários de chegada mais 'nobres', como 'Morning' (Manhã) e 'Evening' (Noite), o que pode indicar rotas com escalas e longa duração.

* 'Late_Night' (Madrugada): Rotas com partida ou chegada de madrugada tendem a ser consistentemente mais baratas, com poucas exceções.

Identificadas as principais cidades no dataset, a análise prossegue para um dos preditores de preço mais intuitivos: a rota (cidade de origem e destino) e, implicitamente, a distância percorrida. Analisar como o preço varia entre as cidades é o próximo passo.

In [ ]:
cities = ['Delhi', 'Mumbai', 'Bangalore', 'Kolkata', 'Hyderabad', 'Chennai']


df_filtered = df_flight[
    df_flight['source_city'].isin(cities) & df_flight['destination_city'].isin(cities)
]


price_matrix = (
    df_filtered.groupby(['source_city', 'destination_city'])['price']
    .mean()
    .unstack()
    .reindex(index=cities, columns=cities)
)


np.fill_diagonal(price_matrix.values, np.nan)


price_matrix.style.background_gradient(cmap="RdYlGn_r", axis=None)\
    .set_caption("Preço Médio das Passagens entre Principais Cidades (₹)")\
    .format("{:.0f}")

A matriz de calor de preços médios revela as rotas mais e menos caras:

* Rotas Mais Caras (Tons Mais Escuros): As passagens de Chennai para Bangalore ($25.082) e de Kolkata para Chennai ($23.660) estão entre as mais caras.

* Rotas Mais Baratas (Tons Mais Claros/Verdes): As rotas de Hyderabad para Delhi ($17.244) e de Delhi para Hyderabad ($17.347) apresentam preços médios mais baixos, sugerindo que a localização geográfica não é o único fator de custo.

Para melhor visualização, vamos analisar também os boxplots das cidades de partida e chegada:

In [ ]:


sns.set(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Paletas discretas ---
palette_source = sns.color_palette("viridis", n_colors=df_flight['source_city'].nunique())
palette_dest = sns.color_palette("magma", n_colors=df_flight['destination_city'].nunique())

# --- Boxplot: Cidade de Origem ---
sns.boxplot(
    data=df_flight,
    x='source_city',
    y='price',
    hue='source_city',
    palette=palette_source,
    legend=False,
    ax=axes[0]
)
axes[0].set_title('Preço por Cidade de Origem', fontsize=13)
axes[0].set_xlabel('Cidade de Origem')
axes[0].set_ylabel('Preço')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, linestyle='--', alpha=0.3)

# --- Boxplot: Cidade de Destino ---
sns.boxplot(
    data=df_flight,
    x='destination_city',
    y='price',
    hue='destination_city',
    palette=palette_dest,
    legend=False,
    ax=axes[1]
)
axes[1].set_title('Preço por Cidade de Destino', fontsize=13)
axes[1].set_xlabel('Cidade de Destino')
axes[1].set_ylabel('')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()


* Origem: Voos partindo de Chennai e Kolkata tendem a ter as medianas de preço mais altas. Em contraste, Hyderabad apresenta a mediana mais baixa, sugerindo que as passagens aéreas originadas nesta cidade tendem a ser mais acessíveis.

* Destino: Voos com destino a Chennai e Bangalore mostram medianas elevadas, indicando que a chegada a esses centros urbanos pode estar associada a custos mais altos.

* Conclusão: A variação do preço é altamente sensível à cidade, tanto na partida quanto na chegada. No entanto, a grande dispersão (outliers na cauda superior) em todas as cidades sugere que o preço final depende crucialmente da combinação de fatores, como a classe da passagem e a companhia aérea, que superam o impacto isolado da localização geográfica.

In [ ]:

city_coords = {
    'Delhi': (28.6139, 77.2090),
    'Mumbai': (19.0760, 72.8777),
    'Bangalore': (12.9716, 77.5946),
    'Kolkata': (22.5726, 88.3639),
    'Hyderabad': (17.3850, 78.4867),
    'Chennai': (13.0827, 80.2707)
}

def haversine_distance(city1, city2):
    from math import radians, sin, cos, sqrt, atan2
    if city1 not in city_coords or city2 not in city_coords:
        return np.nan
    lat1, lon1 = city_coords[city1]
    lat2, lon2 = city_coords[city2]
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


if 'distance' not in df_flight.columns:
    df_flight['distance'] = df_flight.apply(
        lambda row: haversine_distance(row['source_city'], row['destination_city']),
        axis=1
    )


route_stats = (
    df_flight.groupby(['source_city', 'destination_city'])
    .agg({'price': 'mean', 'distance': 'mean'})
    .reset_index()
)
route_stats['src_lat'] = route_stats['source_city'].map(lambda x: city_coords.get(x, (None, None))[0])
route_stats['src_lon'] = route_stats['source_city'].map(lambda x: city_coords.get(x, (None, None))[1])
route_stats['dst_lat'] = route_stats['destination_city'].map(lambda x: city_coords.get(x, (None, None))[0])
route_stats['dst_lon'] = route_stats['destination_city'].map(lambda x: city_coords.get(x, (None, None))[1])


world_path = geodatasets.get_path("naturalearth.land")
world = gpd.read_file(world_path)
india = world.cx[67:98, 6:37]


fig, ax = plt.subplots(figsize=(8, 8))
india.plot(ax=ax, color="#f8f9fa", edgecolor="lightgray", linewidth=0.6)


cmap = get_cmap("RdYlGn_r")  
norm = Normalize(vmin=route_stats['price'].min(), vmax=route_stats['price'].max())


for _, row in route_stats.iterrows():
    color = cmap(norm(row['price']))
    plt.plot(
        [row['src_lon'], row['dst_lon']],
        [row['src_lat'], row['dst_lat']],
        color=color,
        linewidth=2,
        alpha=0.8,
        zorder=2
    )


for city, (lat, lon) in city_coords.items():
    plt.scatter(lon, lat, s=100, color='white', edgecolors='black', linewidth=1.5, zorder=3)
    plt.text(lon + 0.4, lat + 0.3, city, fontsize=9, fontweight='bold', color='#1d3557')


sm = ScalarMappable(norm=norm, cmap=cmap)
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Preço médio da passagem (₹)', fontsize=10)
cbar.ax.tick_params(labelsize=8)


plt.xlim(67, 98)
plt.ylim(6, 33)
plt.title("Mapa da Índia Rotas Aéreas com Heatmap de Preço Médio", fontsize=13, fontweight='bold', pad=15)
plt.axis('off')
plt.tight_layout()
plt.show()


Para verificar a hipótese da distância ser o principal preditor, visualizamos as rotas no mapa real da Índia, onde a cor de cada linha representa o preço médio (vermelho/escuro = mais caro; verde/claro = mais barato).

O mapa desmente a correlação direta entre distância física e preço. Rotas mais curtas podem ser mais caras e vice-versa. Rotas como Chennai para Bangalore (geograficamente próximas) são mais caras do que rotas mais longas, como Hyderabad para Delhi. Isso reforça a ideia de que a demanda do mercado, a competição entre companhias e a qualidade do serviço (classe e número de escalas) são preditores mais importantes do que a distância.

Após analisarmos as correlações numéricas, focamos na relação crítica entre duração do voo (duration) e o preço (price). No entanto, essa relação é fortemente mediada pela companhia aérea.

O gráfico de dispersão a seguir, utilizando uma amostra (df_sample) para maior clareza, nos permite visualizar como o preço se distribui em função da duração do voo, com as cores diferenciando as companhias aéreas. Isso é essencial para identificar se certas empresas dominam as faixas de voos mais longos e caros, revelando a combinação de fatores que impulsiona o custo final.

In [ ]:
sns.scatterplot(
    data=df_sample, x='duration', y='price', hue='airline', alpha=0.6
)
plt.title("Preço vs Duração do Voo por Companhia Aérea")
plt.show()

O scatter plot (acima) reforçou que a Companhia Aérea (airline) é um forte preditor de preço, sendo as companhias de maior custo (como Vistara) associadas a voos com maior duração e faixas de preço elevadas.

Em seguida, focaremos em um fator estrutural que está intimamente ligado tanto à duração quanto à conveniência: o Número de Paradas (stops). A lógica aqui é que menos paradas implicam um voo mais rápido e, teoricamente, mais conveniente, o que deve influenciar o preço. O boxplot a seguir irá quantificar como o preço médio e sua distribuição variam para voos diretos (zero), com uma parada (one) ou com múltiplas paradas (two_or_more).

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df_flight,
    x='stops',
    y='price'
)
plt.title('Preço da Passagem por Número de Paradas', fontsize=13)
plt.xlabel('Número de Paradas')
plt.ylabel('Preço (₹)')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()


Como a análise anterior demonstrou que voos com uma parada (one stop) apresentam a mediana de preço mais alta, é crucial examinar essa variável em um contexto bidimensional. O gráfico de dispersão a seguir cruza Duração do Voo (duration), Preço (price) e Número de Paradas (stops) simultaneamente.

Esta visualização nos permitirá confirmar se a categoria 'one stop' domina os voos mais longos e caros, atuando, na prática, como um proxy para as passagens de Classe Executiva ou Primeira Classe, que frequentemente incluem uma escala em rotas internacionais ou intercontinentais.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df_sample,
    x='duration',
    y='price',
    hue='stops',
    palette='coolwarm',
    alpha=0.6
)
plt.title('Preço vs Duração, colorido pelo Número de Paradas', fontsize=13)
plt.xlabel('Duração (horas)')
plt.ylabel('Preço (₹)')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()



O scatter plot (acima) confirma de forma visualmente poderosa a hipótese levantada:

* Escalas 'one' (laranja/vermelho): Estes voos dominam claramente a faixa de preço mais alta, estendendo-se por quase todo o espectro de duração, inclusive nas viagens mais longas. Isso sugere fortemente que esta categoria absorve a maior parte das passagens de alto valor (como as de classe Business ou First).

* Escalas 'zero' (azul): Embora sejam voos diretos e rápidos, estão restritos a uma faixa de preço muito mais baixa, indicando que são predominantemente passagens de Classe Econômica em rotas regionais.

A partir disso, concluímos que o Número de Paradas é um preditor fortíssimo, mas seu efeito no preço é mais um reflexo da Classe da Passagem e da duração total do que da inconveniência da escala em si.

Até o momento, a análise exploratória permitiu compreender melhor a estrutura e a qualidade do conjunto de dados, destacando as variáveis mais relevantes e suas características gerais. Observou-se a presença de fatores que podem influenciar o preço das passagens, como a companhia aérea, o número de paradas, a duração do voo e a antecedência da compra. Esses padrões iniciais ajudam a delinear hipóteses sobre como cada atributo pode contribuir para a formação do valor final. A partir dessas observações, espera-se que as próximas etapas, envolvendo a regressão e outros modelos preditivos, aprofundem a relação entre as variáveis e permitam identificar quais fatores exercem maior impacto sobre o preço das passagens.